## Task 3 Fundamentals of the Artificial Neural Networks and Backpropagation algorithm I

### Task 3.1 MLP and Backpropagation algorithm

In [3]:
import numpy as np
import sys

In [4]:
#functions of non-linear activations
def f_sigmoid(X, deriv=False):
    if not deriv:
        return 1 / (1 + np.exp(-X))
    else:
        return f_sigmoid(X)*(1 - f_sigmoid(X))

def f_softmax(X):
    Z = np.sum(np.exp(X), axis=1)
    Z = Z.reshape(Z.shape[0], 1)
    return np.exp(X) / Z

def f_relu(X, deriv=False):
    if not deriv:
        return np.maximum(0, X)
    else:
        X[X <= 0] = 0
        X[X > 0] = 1
        return X


In [19]:
def exit_with_err(err_str):
    file = sys.stderr
   # print (file, err_str)
    # sys.exit(1)

In [5]:
def label_to_bit_vector(labels, nbits):
    bit_vector = np.zeros((labels.shape[0], nbits))
    for i in range(labels.shape[0]):
        bit_vector[i, labels[i]] = 1.0

    return bit_vector

In [6]:
#Functionality of a single hidden layer
class Layer:
    def __init__(self, size, batch_size, is_input=False, is_output=False,
                 activation=f_sigmoid):
        self.is_input = is_input
        self.is_output = is_output

        # Z is the matrix that holds output values
        self.Z = np.zeros((batch_size, size[0]))
        # The activation function is an externally defined function (with a
        # derivative) that is stored here
        self.activation = activation

        # W is the outgoing weight matrix for this layer
        self.W = None
        # S is the matrix that holds the inputs to this layer
        self.S = None
        # D is the matrix that holds the deltas for this layer
        self.D = None
        # Fp is the matrix that holds the derivatives of the activation function
        self.Fp = None

        if not is_input:
            self.S = np.zeros((batch_size, size[0]))
            self.D = np.zeros((batch_size, size[0]))

        if not is_output:
            self.W = np.random.normal(size=size, scale=1E-4)

        if not is_input and not is_output:
            self.Fp = np.zeros((size[0], batch_size))

    def forward_propagate(self):
        if self.is_input:
            return self.Z.dot(self.W)

        self.Z = self.activation(self.S)
        if self.is_output:
            return self.Z
        else:
            # For hidden layers, we add the bias values here
            self.Z = np.append(self.Z, np.ones((self.Z.shape[0], 1)), axis=1)
            self.Fp = self.activation(self.S, deriv=True).T
            return self.Z.dot(self.W)


In [7]:
class MultiLayerPerceptron:
    def __init__(self, layer_config, batch_size=100, activation=f_sigmoid):
        self.layers = []
        self.num_layers = len(layer_config)
        self.minibatch_size = batch_size
        self.activation = activation

        for i in range(self.num_layers-1):
            if i == 0:
                print ("Initializing input layer with size {0}.".format(layer_config[i]))
                # Here, we add an additional unit at the input for the bias
                # weight.
                self.layers.append(Layer([layer_config[i]+1, layer_config[i+1]],
                                         batch_size,
                                         is_input=True))
            else:
                print ("Initializing hidden layer with size {0}.".format(layer_config[i]))
                # Here we add an additional unit in the hidden layers for the
                # bias weight.
                self.layers.append(Layer([layer_config[i]+1, layer_config[i+1]],
                                         batch_size,
                                         activation=activation))

        print ("Initializing output layer with size {0}.".format(layer_config[-1]))
        self.layers.append(Layer([layer_config[-1], None],
                                 batch_size,
                                 is_output=True,
                                 activation=f_softmax))
        print ("Done!")

    def forward_propagate(self, data):
        # We need to be sure to add bias values to the input
        self.layers[0].Z = np.append(data, np.ones((data.shape[0], 1)), axis=1)

        for i in range(self.num_layers-1):
            self.layers[i+1].S = self.layers[i].forward_propagate()
        return self.layers[-1].forward_propagate()

    def backpropagate(self, yhat, labels):

        exit_with_err("FIND ME IN THE CODE, What is computed in the next line of code?\n")
        # What's been computed in the next line of code is delta for the output layer, which is the difference between predicted output and actual labels.
        
        self.layers[-1].D = (yhat - labels).T

        for i in range(self.num_layers-2, 0, -1):
            # We do not calculate deltas for the bias values
            W_nobias = self.layers[i].W[0:-1, :]

            exit_with_err("FIND ME IN THE CODE, What does this 'for' loop do?\n")
            # This 'for' loop computes the deltas for each hidden layer by propagating the error backward through the network using the weights and the derivative of the activation function.
            
            for i in range(self.num_layers-2, 0, -1):
                # We do not calculate deltas for the bias values
                W_nobias = self.layers[i].W[0:-1, :]
                self.layers[i].D = W_nobias.dot(self.layers[i+1].D) * self.layers[i].Fp

    def update_weights(self, eta):
        
        for i in range(0, self.num_layers-1):
            W_grad = -eta*(self.layers[i+1].D.dot(self.layers[i].Z)).T
            self.layers[i].W += W_grad

    def evaluate(self, train_data, train_labels, test_data, test_labels,
                 num_epochs=70, eta=0.05, eval_train=False, eval_test=True):

        N_train = len(train_labels)*len(train_labels[0])
        N_test = len(test_labels)*len(test_labels[0])

        print ("Training for {0} epochs...".format(num_epochs))
        for t in range(0, num_epochs):
            out_str = "[{0:4d}] ".format(t)

            for b_data, b_labels in zip(train_data, train_labels):
                output = self.forward_propagate(b_data)
                self.backpropagate(output, b_labels)

                exit_with_err("FIND ME IN THE CODE, How does weight update is implemented? What is eta?\n")
                # The weight update is implemented by calculating the gradient of the loss with respect to the weights using the deltas and the outputs of the previous layer. 
                # The weights are then updated as follows: New Weights = Old Weights - eta * Gradient
                # Here, eta is the learning rate, which controls the step size during the weight update
        
                self.update_weights(eta=eta)

            if eval_train:
                errs = 0
                for b_data, b_labels in zip(train_data, train_labels):
                    output = self.forward_propagate(b_data)
                    yhat = np.argmax(output, axis=1)
                    errs += np.sum(1-b_labels[np.arange(len(b_labels)), yhat])

                out_str = ("{0} Training error: {1:.5f}".format(out_str,
                                                           float(errs)/N_train))

            if eval_test:
                errs = 0
                for b_data, b_labels in zip(test_data, test_labels):
                    output = self.forward_propagate(b_data)
                    yhat = np.argmax(output, axis=1)
                    errs += np.sum(1-b_labels[np.arange(len(b_labels)), yhat])

                out_str = ("{0} Test error: {1:.5f}").format(out_str,
                                                       float(errs)/N_test)

            print (out_str)


In [7]:
def create_batches(data, labels, batch_size, create_bit_vector=False):
    N = data.shape[0]
    print ("Batch size {0}, the number of examples {1}.".format(batch_size,N))

    if N % batch_size != 0:
        print ("Warning in create_minibatches(): Batch size {0} does not " \
              "evenly divide the number of examples {1}.".format(batch_size,N))
    chunked_data = []
    chunked_labels = []
    idx = 0
    while idx + batch_size <= N:
        chunked_data.append(data[idx:idx+batch_size, :])
        if not create_bit_vector:
            chunked_labels.append(labels[idx:idx+batch_size])
        else:
            bit_vector = label_to_bit_vector(labels[idx:idx+batch_size], 10)
            chunked_labels.append(bit_vector)

        idx += batch_size

    return chunked_data, chunked_labels


In [8]:
def prepare_for_backprop(batch_size, Train_images, Train_labels, Valid_images, Valid_labels):

    print ("Creating data...")
    batched_train_data, batched_train_labels = create_batches(Train_images, Train_labels,
                                              batch_size,
                                              create_bit_vector=True)
    batched_valid_data, batched_valid_labels = create_batches(Valid_images, Valid_labels,
                                              batch_size,
                                              create_bit_vector=True)
    print ("Done!")


    return batched_train_data, batched_train_labels,  batched_valid_data, batched_valid_labels



In [9]:
from keras.datasets import mnist

In [10]:
(Xtr, Ltr), (X_test, L_test)=mnist.load_data()

Xtr = Xtr.reshape(60000, 784)
X_test = X_test.reshape(10000, 784)
Xtr = Xtr.astype('float32')
X_test = X_test.astype('float32')
Xtr /= 255
X_test /= 255
print(Xtr.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
60000 train samples
10000 test samples


In [20]:
# Backprop with default configuration

batch_size=100

train_data, train_labels, valid_data, valid_labels=prepare_for_backprop(batch_size, Xtr, Ltr, X_test, L_test)

mlp = MultiLayerPerceptron(layer_config=[784, 100, 100, 10], batch_size=batch_size, activation=f_sigmoid)

mlp.evaluate(train_data, train_labels, valid_data, valid_labels, num_epochs=70, eta=0.05,
             eval_train=True)

print("Done:)\n")


Creating data...
Batch size 100, the number of examples 60000.
Batch size 100, the number of examples 10000.
Done!
Initializing input layer with size 784.
Initializing hidden layer with size 100.
Initializing hidden layer with size 100.
Initializing output layer with size 10.
Done!
Training for 70 epochs...
[   0]  Training error: 0.37697 Test error: 0.37900
[   1]  Training error: 0.13915 Test error: 0.13590
[   2]  Training error: 0.05833 Test error: 0.05850
[   3]  Training error: 0.04547 Test error: 0.04890
[   4]  Training error: 0.04380 Test error: 0.04720
[   5]  Training error: 0.03030 Test error: 0.04090
[   6]  Training error: 0.02763 Test error: 0.03700
[   7]  Training error: 0.02697 Test error: 0.03810
[   8]  Training error: 0.02258 Test error: 0.03440
[   9]  Training error: 0.02203 Test error: 0.03520
[  10]  Training error: 0.02333 Test error: 0.03700
[  11]  Training error: 0.03340 Test error: 0.04640
[  12]  Training error: 0.01790 Test error: 0.03350
[  13]  Trainin

Running the code with the suggested default configurated hyperparameters: epochs = 70, and learning rate = 0.05, we obtained a test error of 0.02670, which gives us an accuracy of 97.33%.

In [ ]:
# Altered experiment with lower learning rate

print("\n=== Training with Low Learning Rate ===")
print("Activation: Sigmoid, Learning rate: 0.005, Epochs: 70")

mlp_low_lr = MultiLayerPerceptron(
    layer_config=[784, 100, 100, 10],
    batch_size=batch_size,
    activation=f_sigmoid
)

mlp_low_lr.evaluate(train_data, train_labels, valid_data, valid_labels,
                    num_epochs=70, eta=0.005, eval_train=True)




=== Training with Low Learning Rate ===
Activation: Sigmoid, Learning rate: 0.005, Epochs: 70
Initializing input layer with size 784.
Initializing hidden layer with size 100.
Initializing hidden layer with size 100.
Initializing output layer with size 10.
Done!
Training for 70 epochs...
[   0]  Training error: 0.70337 Test error: 0.70080
[   1]  Training error: 0.64712 Test error: 0.64340
[   2]  Training error: 0.59890 Test error: 0.59870
[   3]  Training error: 0.46413 Test error: 0.47380
[   4]  Training error: 0.20988 Test error: 0.20090
[   5]  Training error: 0.11377 Test error: 0.10940
[   6]  Training error: 0.09045 Test error: 0.08870
[   7]  Training error: 0.07613 Test error: 0.07620
[   8]  Training error: 0.06495 Test error: 0.06460
[   9]  Training error: 0.05562 Test error: 0.05650
[  10]  Training error: 0.04902 Test error: 0.04980
[  11]  Training error: 0.04387 Test error: 0.04480
[  12]  Training error: 0.03900 Test error: 0.04220
[  13]  Training error: 0.03592 Tes

When we lowered the learning rate to 0.005 while keeping the number of epochs the same (70), we obained a test error  of 0.02660, resulting in an accuracy of 97.34%. This may indicate that a lower learning rate doesn't necessarily lead to better performance. It can lead to slower convergence and potentially getting stuck in local minima, which may negatively impact the model's performance.

In [22]:
# Altered experiment with higher learning rate

print("\n=== Training with High Learning Rate ===")
print("Activation: Sigmoid, Learning rate: 0.5, Epochs: 70")

mlp_high_lr = MultiLayerPerceptron(
    layer_config=[784, 100, 100, 10],
    batch_size=batch_size,
    activation=f_sigmoid
)

mlp_high_lr.evaluate(train_data, train_labels, valid_data, valid_labels,
                     num_epochs=70, eta=0.5, eval_train=True)

print("\nObservation: With eta=0.5, learning may be unstable.")
print("The error may oscillate or diverge because steps are too large.")
print("This demonstrates the importance of choosing an appropriate learning rate.")


=== Training with High Learning Rate ===
Activation: Sigmoid, Learning rate: 0.5, Epochs: 70
Initializing input layer with size 784.
Initializing hidden layer with size 100.
Initializing hidden layer with size 100.
Initializing output layer with size 10.
Done!
Training for 70 epochs...
[   0]  Training error: 0.90263 Test error: 0.90180
[   1]  Training error: 0.90128 Test error: 0.90200
[   2]  Training error: 0.89782 Test error: 0.89900
[   3]  Training error: 0.90070 Test error: 0.89680
[   4]  Training error: 0.90128 Test error: 0.90200
[   5]  Training error: 0.88763 Test error: 0.88650
[   6]  Training error: 0.89558 Test error: 0.89720
[   7]  Training error: 0.90085 Test error: 0.89910
[   8]  Training error: 0.88763 Test error: 0.88650
[   9]  Training error: 0.90248 Test error: 0.90260
[  10]  Training error: 0.90085 Test error: 0.89910
[  11]  Training error: 0.90128 Test error: 0.90200
[  12]  Training error: 0.90248 Test error: 0.90260
[  13]  Training error: 0.88763 Test

In [23]:
# Altered experiemnt with ReLU activation

print("\n=== Training with ReLU Activation ===")
print("Activation: ReLU, Learning rate: 0.05, Epochs: 70")

mlp_relu = MultiLayerPerceptron(
    layer_config=[784, 100, 100, 10],
    batch_size=batch_size,
    activation=f_relu
)

mlp_relu.evaluate(train_data, train_labels, valid_data, valid_labels,
                  num_epochs=70, eta=0.05, eval_train=True)

print("\nReLU may need different learning rates to perform optimally.")


=== Training with ReLU Activation ===
Activation: ReLU, Learning rate: 0.05, Epochs: 70
Initializing input layer with size 784.
Initializing hidden layer with size 100.
Initializing hidden layer with size 100.
Initializing output layer with size 10.
Done!
Training for 70 epochs...
[   0]  Training error: 0.90137 Test error: 0.90420
[   1]  Training error: 0.90137 Test error: 0.90420
[   2]  Training error: 0.90137 Test error: 0.90420
[   3]  Training error: 0.90137 Test error: 0.90420
[   4]  Training error: 0.90137 Test error: 0.90420
[   5]  Training error: 0.90137 Test error: 0.90420
[   6]  Training error: 0.90137 Test error: 0.90420
[   7]  Training error: 0.90137 Test error: 0.90420
[   8]  Training error: 0.90137 Test error: 0.90420
[   9]  Training error: 0.90137 Test error: 0.90420
[  10]  Training error: 0.90137 Test error: 0.90420
[  11]  Training error: 0.90137 Test error: 0.90420
[  12]  Training error: 0.90137 Test error: 0.90420
[  13]  Training error: 0.90137 Test erro

#### Testing different learning rates with RELU activation function

In [26]:
# Test different learning rates with ReLU
print("\n=== Finding Optimal Learning Rate for ReLU ===")

learning_rates_relu = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.5]

for lr in learning_rates_relu:
    print(f"\nTesting ReLU with learning rate: {lr}")

    mlp_relu_test = MultiLayerPerceptron(
        layer_config=[784, 100, 100, 10],
        batch_size=batch_size,
        activation=f_relu
    )

    mlp_relu_test.evaluate(train_data, train_labels, valid_data, valid_labels,
                           num_epochs=70, eta=lr, eval_train=False, eval_test=True)

print("\nConclusion: ReLU typically requires lower learning rates than Sigmoid.")
print("This is because ReLU gradients don't vanish like sigmoid, leading to larger updates.")


=== Finding Optimal Learning Rate for ReLU ===

Testing ReLU with learning rate: 0.0001
Initializing input layer with size 784.
Initializing hidden layer with size 100.
Initializing hidden layer with size 100.
Initializing output layer with size 10.
Done!
Training for 70 epochs...
[   0]  Test error: 0.88650
[   1]  Test error: 0.88650
[   2]  Test error: 0.88650
[   3]  Test error: 0.88650
[   4]  Test error: 0.88650
[   5]  Test error: 0.88650
[   6]  Test error: 0.88650
[   7]  Test error: 0.88650
[   8]  Test error: 0.88650
[   9]  Test error: 0.88650
[  10]  Test error: 0.88650
[  11]  Test error: 0.88650
[  12]  Test error: 0.88650
[  13]  Test error: 0.88650
[  14]  Test error: 0.88650
[  15]  Test error: 0.88650
[  16]  Test error: 0.88650
[  17]  Test error: 0.88650
[  18]  Test error: 0.88650
[  19]  Test error: 0.88650
[  20]  Test error: 0.88650
[  21]  Test error: 0.88650
[  22]  Test error: 0.88650
[  23]  Test error: 0.88650
[  24]  Test error: 0.88650
[  25]  Test erro

After conducting several tests, we discovered that the learning rate that gives us the same results as the default configuration for the MLP with Sigmoid activation function, but for ReLU is 0.005, which yields a similar test error to 0.02670 during the last phases of training. The final resulting test error was 0.01990, resulting in an accuracy of 98.01%.